In [1]:
5+10

15

# Downstream Meta Checkpoint Sweep

Evaluate the three downstream meta verifier checkpoints under the same attack suite used by `[3.a] attack_eval_our_improved_meta.ipynb`.

In [2]:
from types import SimpleNamespace
from pathlib import Path
import time

import pandas as pd
import torch
from torch.utils.data import DataLoader

from eval_downstream_meta_checkpoints import (
    build_validation_dataset,
    evaluate_checkpoint,
    load_pipe,
    set_seed,
)
from watermark import get_watermarking_mask, get_watermarking_pattern

## Config

In [3]:
args = SimpleNamespace(
    data_dir="./verifier_dataset_stablediff_octoweb",
    output_dir="./eval_results/downstream_meta_checkpoint_sweep",
    checkpoints=[
        "verifier_dataset_stablediff_octoweb_downstream_from_nvidia_meta_iter2000_300_epoch116.pth",
        "verifier_dataset_stablediff_octoweb_downstream_from_nvidia_meta_iter2000_300_epoch110.pth",
        "verifier_dataset_stablediff_octoweb_downstream_from_nvidia_meta_iter2000_300_final.pth",
    ],
    model_id="Manojb/stable-diffusion-2-1-base",
    batch_size=8,
    num_workers=0,
    testing_times=5,
    validation_split=0.15,
    seed=42,
    image_size=512,
    num_inference_steps=50,
    guidance_scale=7.5,
    w_mask_shape="circle",
    w_channel=0,
    w_radius=10,
    w_strength=0.99,
    w_pattern="octoweb",
    cpu=False,
)

device = "cpu" if args.cpu else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Checkpoints:")
for ckpt in args.checkpoints:
    print(" -", ckpt)

Device: cuda
Checkpoints:
 - verifier_dataset_stablediff_octoweb_downstream_from_nvidia_meta_iter2000_300_epoch116.pth
 - verifier_dataset_stablediff_octoweb_downstream_from_nvidia_meta_iter2000_300_epoch110.pth
 - verifier_dataset_stablediff_octoweb_downstream_from_nvidia_meta_iter2000_300_final.pth


## Setup Pipeline And Validation Dataset

In [4]:
set_seed(args.seed)

missing = [path for path in args.checkpoints if not Path(path).exists()]
if missing:
    raise FileNotFoundError(f"Missing checkpoint(s): {missing}")

Path(args.output_dir).mkdir(parents=True, exist_ok=True)

pipe, text_embeddings = load_pipe(args, device)

watermarking_mask = get_watermarking_mask(
    pipe.get_random_latents(),
    w_mask_shape=args.w_mask_shape,
    w_channel=args.w_channel,
    w_radius=args.w_radius,
    device=device,
)

gt_patch = get_watermarking_pattern(
    pipe,
    w_seed=args.seed,
    w_pattern=args.w_pattern,
    w_radius=args.w_radius,
    device=device,
    strength=args.w_strength,
    shape=None,
)

dataset = build_validation_dataset(
    args, pipe, text_embeddings, watermarking_mask, gt_patch, device
)

loader = DataLoader(
    dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=args.num_workers,
)

print("Ready. Validation dataset size:", len(dataset))

Keyword arguments {'verbose': False} are not expected by InversableStableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Validation samples: 150 / 1000
First validation labels: [1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0]
Ready. Validation dataset size: 150


d:\GitHub\watermarking\inverse_stable_diffusion.py:87: FutureWarning: Accessing config attribute `in_channels` directly via 'UNet2DConditionModel' object attribute is deprecated. Please access 'in_channels' over 'UNet2DConditionModel's config object instead, e.g. 'unet.config.in_channels'.
  num_channels_latents = self.unet.in_channels
d:\GitHub\watermarking\watermark.py:471: UserWarning: ComplexHalf support is experimental and many operators don't support it yet. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\EmptyTensor.cpp:58.)
  return gt_patch.to(dtype=torch.complex32)


## Run Evaluation

In [5]:
all_dfs = []
total_start = time.time()

for checkpoint_path in args.checkpoints:
    df = evaluate_checkpoint(checkpoint_path, args, pipe, dataset, loader, device)
    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)
combined_csv = Path(args.output_dir) / "combined_attack_eval_summary.csv"
combined.to_csv(combined_csv, index=False)

print("Combined summary:", combined_csv)
print(f"Total elapsed minutes: {(time.time() - total_start) / 60:.2f}")
combined

c:\Users\chihc\miniconda3\envs\pytorch\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\chihc\miniconda3\envs\pytorch\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



Evaluating checkpoint: verifier_dataset_stablediff_octoweb_downstream_from_nvidia_meta_iter2000_300_epoch116.pth
Output dir: eval_results\downstream_meta_checkpoint_sweep\epoch116
Using PNSRL1 wrapper for the model.
Loading checkpoint: verifier_dataset_stablediff_octoweb_downstream_from_nvidia_meta_iter2000_300_epoch116.pth
Restored model and optimizer. Resuming from epoch 117


clean | Testing runs:   0%|          | 0/5 [00:00<?, ?it/s]d:\GitHub\watermarking\ds.py:153: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Copy.cpp:309.)
  mse = (diff.float() ** 2).mean().clamp_min(eps)
clean | Testing runs: 100%|██████████| 5/5 [24:37<00:00, 295.47s/it]


    epoch116 | clean          our_acc=0.9800 our_auc=0.9995 l1_acc=0.9867 psnr_acc=0.9800


jpeg_strong | Testing runs: 100%|██████████| 5/5 [24:58<00:00, 299.80s/it]


...

geom_warp | Testing runs: 100%|██████████| 5/5 [26:05<00:00, 313.08s/it]


       final | geom_warp      our_acc=0.8480 our_auc=0.9427 l1_acc=0.5680 psnr_acc=0.5307


train_aug_mix | Testing runs: 100%|██████████| 5/5 [26:20<00:00, 316.09s/it]

       final | train_aug_mix  our_acc=0.7480 our_auc=0.8389 l1_acc=0.5333 psnr_acc=0.5187
Saved checkpoint summary: eval_results\downstream_meta_checkpoint_sweep\final\attack_eval_summary.csv
Combined summary: eval_results\downstream_meta_checkpoint_sweep\combined_attack_eval_summary.csv
Total elapsed minutes: 566.85


,checkpoint_label,checkpoint,attack,our_acc,our_auc,best_our_thr,l1_acc,l1_auc,best_l1_thr,psnr_acc,psnr_auc,best_psnr_thr,clean_best_l1_thr,clean_best_psnr_thr,elapsed_sec
0,epoch116,verifier_dataset_stablediff_octoweb_downstream...,clean,0.980000,0.999465,0.996387,0.986667,0.998930,-45.78125,0.980000,0.995899,-17.788549,-45.78125,-17.788549,1477.354680
1,epoch116,verifier_dataset_stablediff_octoweb_downstream...,jpeg_strong,0.934667,0.986657,0.671122,0.644000,0.952252,-52.31250,0.609333,0.888707,-19.276882,-45.78125,-17.788549,1498.995419
2,epoch116,verifier_dataset_stablediff_octoweb_downstream...,msg_app_combo,0.917333,0.955800,0.470913,0.480000,0.870198,-56.78125,0.476000,0.728978,-20.411182,-45.78125,-17.788549,877.797375
3,epoch116,verifier_dataset_stablediff_octoweb_downstream...,down_up,0.953333,0.993938,0.686295,0.733333,0.985648,-53.75000,0.573333,0.943305,-19.649616,-45.78125,-17.788549,876.365754
4,epoch116,verifier_dataset_stablediff_octoweb_downstream...,blur,0.904000,0.967752,0.609815,0.500000,0.886311,-55.25000,0.478667,0.800121,-20.251005,-45.78125,-17.788549,881.555264
5,epoch116,verifier_dataset_stablediff_octoweb_downstream...,random_crop,0.902667,0.969107,0.511982,0.624000,0.975842,-52.21875,0.560000,0.947627,-19.271465,-45.78125,-17.788549,877.165881
6,epoch116,verifier_dataset_stablediff_octoweb_downstream...,occlusion,0.980000,0.999280,0.985542,0.973333,0.998224,-48.53125,0.946667,0.994138,-18.205666,-45.78125,-17.788549,879.964871
7,epoch116,verifier_dataset_stablediff_octoweb_downstream...,geom_warp,0.889333,0.962610,0.603340,0.578667,0.972430,-52.34375,0.537333,0.905438,-19.402058,-45.78125,-17.788549,891.645483
8,epoch116,verifier_dataset_stablediff_octoweb_downstream...,train_aug_mix,0.802667,0.883017,0.523349,0.529333,0.865819,-53.90625,0.518667,0.785053,-19.363953,-45.78125,-17.788549,923.096931
9,epoch110,verifier_dataset_stablediff_octoweb_downstream...,clean,0.980000,0.999465,0.997294,0.986667,0.998930,-45.78125,0.980000,0.995899,-17.788549,-45.78125,-17.788549,1186.181900


## Accuracy Ranking

In [6]:
ranking = combined.sort_values(
    ["attack", "our_acc"], ascending=[True, False]
)[["attack", "checkpoint_label", "our_acc", "our_auc", "l1_acc", "psnr_acc"]]
ranking

,attack,checkpoint_label,our_acc,our_auc,l1_acc,psnr_acc
22,blur,final,0.934667,0.973229,0.508000,0.477333
4,blur,epoch116,0.904000,0.967752,0.500000,0.478667
13,blur,epoch110,0.870667,0.971788,0.505333,0.477333
0,clean,epoch116,0.980000,0.999465,0.986667,0.980000
9,clean,epoch110,0.980000,0.999465,0.986667,0.980000
18,clean,final,0.946667,0.999287,0.986667,0.980000
3,down_up,epoch116,0.953333,0.993938,0.733333,0.573333
12,down_up,epoch110,0.946667,0.994473,0.733333,0.573333
21,down_up,final,0.946667,0.993225,0.733333,0.573333
7,geom_warp,epoch116,0.889333,0.962610,0.578667,0.537333


## Mean Accuracy By Checkpoint

In [7]:
mean_scores = (
    combined.groupby("checkpoint_label", as_index=False)
    .agg(mean_our_acc=("our_acc", "mean"), mean_our_auc=("our_auc", "mean"))
    .sort_values("mean_our_acc", ascending=False)
)
mean_scores

,checkpoint_label,mean_our_acc,mean_our_auc
1,epoch116,0.918222,0.968625
0,epoch110,0.907556,0.966177
2,final,0.895852,0.954462
